# 03 · Baseline Analysis

This notebook reads the released baseline reports in `examples/` and analyses them across the
five tasks: primary metrics, four-dimensional stratification, hard subsets, and significance
against the reference baseline.

**Prerequisites.** One of:

```bash
# real data (recommended)
python scripts/run_all_baselines.py --mode small
python scripts/build_leaderboard.py

# offline / synthetic
python scripts/generate_synthetic.py --small
python scripts/build_dataset.py --source synthetic
python scripts/run_all_baselines.py --mode small
```

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLES = ROOT / "examples"
print("examples dir:", EXAMPLES, "exists:", EXAMPLES.exists())

## 1. Leaderboard

The aggregated leaderboard produced by `scripts/build_leaderboard.py`.
Note that `--mode small` samples the test window, so these numbers are indicative rather than
final; the paper and the public leaderboard use the full rolling-window run.

In [ ]:
csv_path = EXAMPLES / "leaderboard.csv"
if csv_path.exists():
    board = pd.read_csv(csv_path)
    display(board.sort_values(["task", "_rank"]) if "_rank" in board.columns else board)
else:
    board = pd.DataFrame()
    print("leaderboard.csv not found - run scripts/build_leaderboard.py")

## 2. Per-task primary metric

One row per (task, model) with the primary metric value and the number of evaluated samples.

In [ ]:
reports = []
for path in sorted(EXAMPLES.glob("*__*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    overall = payload.get("overall", {}) or {}
    reports.append(
        {
            "file": path.name,
            "task": payload.get("task"),
            "model": payload.get("model"),
            "kind": (payload.get("cost") or {}).get("baseline_kind"),
            "n": overall.get("n"),
            "failed": bool((payload.get("extras") or {}).get("failed", False)),
            "has_significance": bool(payload.get("significance")),
            "overall_keys": ",".join(sorted(k for k in overall if k != "events")),
        }
    )

table = pd.DataFrame(reports)
display(table)
print("reports found:", len(table))

## 3. Stratified breakdown

Four-dimensional stratification is mandatory in the protocol: by event, skill level, time slice,
and continent. Below we inspect the `by_event` stratum of one report.

In [ ]:
def pick(preferred: str) -> Path | None:
    exact = EXAMPLES / preferred
    if exact.exists():
        return exact
    matches = sorted(EXAMPLES.glob(preferred.replace("__*", "__*").split("__")[0] + "__*.json"))
    return matches[0] if matches else None


target = pick("result_prediction__xgboost_log.json") or pick("dnf__historical_dnf_rate.json")
print("inspecting:", target.name if target else None)

if target:
    payload = json.loads(target.read_text(encoding="utf-8"))
    strata = payload.get("stratified") or {}
    print("available strata:", list(strata))
    by_event = strata.get("by_event") or {}
    if by_event:
        display(pd.DataFrame(by_event).T)
    print("\nhard_subset:", json.dumps(payload.get("hard_subset") or {}, ensure_ascii=False)[:400])
    print("\nsignificance:", json.dumps(payload.get("significance") or {}, ensure_ascii=False)[:400])

## 4. Cross-task coverage

Coverage check against the acceptance criterion *≥3 baselines per task*.

In [ ]:
if not table.empty:
    coverage = (
        table[~table["failed"]]
        .groupby("task")["model"]
        .agg(models="nunique", names=lambda s: ", ".join(sorted(s)))
        .sort_values("models", ascending=False)
    )
    coverage["meets_>=3"] = coverage["models"] >= 3
    display(coverage)
    print("total models:", int(coverage["models"].sum()))

## 5. Optional plot

A quick bar chart of the primary metric per model, per task.

In [ ]:
try:
    import matplotlib.pyplot as plt

    if not board.empty and "primary_value" in board.columns:
        tasks = sorted(board["task"].dropna().unique())
        fig, axes = plt.subplots(1, len(tasks), figsize=(4 * len(tasks), 3.2), squeeze=False)
        for ax, task in zip(axes[0], tasks):
            sub = board[board["task"] == task].sort_values("primary_value")
            ax.barh(sub["model"].astype(str), sub["primary_value"].astype(float))
            ax.set_title(task, fontsize=10)
            ax.tick_params(labelsize=8)
        fig.tight_layout()
        plt.show()
    else:
        print("no leaderboard data to plot")
except ImportError:
    print("matplotlib not installed - skipping plot (pip install matplotlib)")